# 15.8 — PDE ellittiche

**Prerequisiti:** 15.7 Equazione di Poisson, 14.7 Teorema di Hahn-Banach, 6.17 Autovalori.


## Perché ci interessa

Nel notebook 15.7, il teorema di Riesz ha risolto $-\Delta u=f$ sfruttando il fatto che la forma bilineare $\int\nabla u\cdot\nabla v$ è **simmetrica** (coincide con un prodotto scalare). Molte equazioni fisicamente rilevanti — con termini di trasporto, o coefficienti variabili non simmetrici — danno forme bilineari **non simmetriche**, dove Riesz non si applica direttamente. Il **teorema di Lax-Milgram**, generalizzazione del teorema di Riesz, copre esattamente questo caso, definendo la classe generale delle equazioni **ellittiche** (di cui Laplace e Poisson sono il caso modello) e fornendo lo strumento astratto per dimostrarne esistenza e unicità in un colpo solo.

## Teoria, passo dopo passo

### Operatori ellittici in forma di divergenza

Un operatore differenziale del secondo ordine in forma di divergenza è $Lu=-\operatorname{div}(A(x)\nabla u)+b(x)\cdot\nabla u+c(x)u$, con $A(x)$ una matrice $n\times n$ per ogni $x$. $L$ è **ellittico (uniformemente)** se esiste $\alpha>0$ tale che $\xi^TA(x)\xi\ge\alpha|\xi|^2$ per ogni $x\in\Omega$, $\xi\in\mathbb R^n$ (una condizione di "positività uniforme" sulla matrice $A$, che generalizza la condizione che gli **autovalori** di $A(x)$, notebook 6.17, siano tutti $\ge\alpha>0$). Per $A=I$ (matrice identità), $b=0$, $c=0$: $Lu=-\Delta u$, il caso del notebook 15.7.

### Forme bilineari associate

Come nel notebook 15.7, moltiplicando per $v\in H^1_0(\Omega)$ e integrando per parti: $B(u,v)=\int_\Omega\big[A(x)\nabla u\cdot\nabla v+(b\cdot\nabla u)v+cuv\big]\,dx$. A differenza di $\int\nabla u\cdot\nabla v$ (simmetrica in $u,v$), $B$ è in generale **non simmetrica** se $b\ne0$ (il termine $(b\cdot\nabla u)v$ non è simmetrico scambiando $u,v$).

### Il teorema di Lax-Milgram

**Teorema (Lax-Milgram).** Sia $H$ uno spazio di Hilbert, $B:H\times H\to\mathbb R$ una forma bilineare **continua** ($|B(u,v)|\le C\|u\|\|v\|$) e **coerciva** ($B(u,u)\ge\alpha\|u\|^2$ per un $\alpha>0$, per ogni $u$), e $\varphi\in H^*$ un funzionale lineare continuo. Allora esiste un **unico** $u\in H$ tale che $B(u,v)=\varphi(v)$ per ogni $v\in H$.

**Dimostrazione (schema, riducendosi a Riesz).** Per $u$ fissato, $v\mapsto B(u,v)$ è un funzionale lineare continuo (per la continuità di $B$); per Riesz (notebook 14.6), esiste un unico $Tu\in H$ con $B(u,v)=\langle Tu,v\rangle$ per ogni $v$: questo definisce un operatore $T:H\to H$, lineare (per bilinearità di $B$) e limitato (con $\|T\|\le C$, dalla continuità di $B$). L'equazione $B(u,v)=\varphi(v)$ diventa $\langle Tu,v\rangle=\langle w,v\rangle$ per ogni $v$ (dove $w\in H$ rappresenta $\varphi$ secondo Riesz), cioè $Tu=w$: dobbiamo dimostrare che $T$ è **invertibile**. La coercività dà $\|Tu\|\|u\|\ge\langle Tu,u\rangle=B(u,u)\ge\alpha\|u\|^2$ (Cauchy-Schwarz sul primo passaggio), quindi $\|Tu\|\ge\alpha\|u\|$: $T$ è **iniettivo** e con **immagine chiusa** (se $Tu_n\to z$, allora $(u_n)$ è di Cauchy per la stima appena vista, converge a un $u$ per completezza di $H$, e per continuità $Tu=z$). Si dimostra poi (usando un argomento collegato al teorema di Hahn-Banach, notebook 14.7, applicato al complemento ortogonale dell'immagine, che risulta necessariamente banale) che $T$ è anche **suriettivo**. Essendo $T$ biettivo con $\|T^{-1}\|\le1/\alpha$ (dalla stima $\|Tu\|\ge\alpha\|u\|$), esiste un unico $u=T^{-1}w$. $\blacksquare$

**Lax-Milgram generalizza Riesz.** Se $B$ è **simmetrica**, $B(u,v)=\langle u,v\rangle_B$ è essa stessa un prodotto scalare (equivalente a quello originale, per coercività e continuità), e Lax-Milgram si riduce esattamente al teorema di Riesz applicato a questo nuovo prodotto scalare — coerentemente con il fatto che il notebook 15.7 (caso simmetrico, $b=0$) usava Riesz direttamente.

### Buona posizione per operatori ellittici generali

**Corollario (esistenza e unicità per PDE ellittiche generali).** Se $L$ è uniformemente ellittico e $c(x)\ge0$ (una condizione tecnica che garantisce la coercività di $B$, verificabile stimando $B(u,u)\ge\alpha\|\nabla u\|_2^2+\int cu^2\,dx-\|b\|_\infty\|\nabla u\|_2\|u\|_2\ge\alpha\|\nabla u\|_2^2-\varepsilon\|\nabla u\|_2^2-C_\varepsilon\|u\|_2^2$ per una disuguaglianza di Young opportuna, poi usando Poincaré, notebook 15.7, per assorbire il termine $\|u\|_2^2$), il problema $Lu=f$ in $\Omega$, $u=0$ su $\partial\Omega$, ha un'unica soluzione debole in $H^1_0(\Omega)$ per ogni $f\in L^2(\Omega)$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# Risolviamo numericamente un problema ellittico NON simmetrico: -u'' + b*u' + c*u = f su (0,1), u(0)=u(1)=0
# (versione 1D, dove il termine di trasporto b*u' rompe la simmetria)

N = 200
h = 1/(N+1)
x = np.linspace(0, 1, N+2)[1:-1]

b_coeff = 5.0  # termine di trasporto (rompe la simmetria)
c_coeff = 1.0

def f(x):
    return np.ones_like(x)

# Discretizzazione: -u'' con differenze centrate, u' con differenze centrate, u con termine diretto
main_diag = (2/h**2 + c_coeff) * np.ones(N)
lower_diag = (-1/h**2 - b_coeff/(2*h)) * np.ones(N-1)
upper_diag = (-1/h**2 + b_coeff/(2*h)) * np.ones(N-1)

A = diags([lower_diag, main_diag, upper_diag], offsets=[-1, 0, 1]).tocsr()
b_vec = f(x)

u_numerico = spsolve(A, b_vec)

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(x, u_numerico, 'b-')
ax.set_title(f"Soluzione di -u''+{b_coeff}u'+{c_coeff}u=1, u(0)=u(1)=0 (operatore NON simmetrico)")
plt.tight_layout()
plt.show()

print("Il termine di trasporto b*u' rende la matrice A non simmetrica:")
print("A simmetrica?", np.allclose(A.toarray(), A.toarray().T))
print("Nonostante questo, Lax-Milgram garantisce esistenza e unicita' (per c>=0 e b limitato)")

## Esercizio svolto e commentato

**Problema:** verifica la coercività della forma bilineare $B(u,v)=\int_0^1(u'v'+u'v+uv)\,dx$ su $H^1_0(0,1)$ (corrispondente all'operatore $-u''+u'+u$), usando la disuguaglianza di Young e la disuguaglianza di Poincaré del notebook 15.7.

**Ragionamento:**

**Passo 1 — scriviamo $B(u,u)$.** $B(u,u)=\int_0^1\big[(u')^2+u'u+u^2\big]\,dx=\|u'\|_2^2+\int_0^1u'u\,dx+\|u\|_2^2$.

**Passo 2 — stimiamo il termine misto con la disuguaglianza di Young.** $\Big|\int_0^1u'u\,dx\Big|\le\|u'\|_2\|u\|_2$ (Cauchy-Schwarz, notebook 14.3) $\le\dfrac12\|u'\|_2^2+\dfrac12\|u\|_2^2$ (disuguaglianza di Young $ab\le\frac12a^2+\frac12b^2$, caso particolare di quella usata nel notebook 13.8 per Hölder). Dunque $\int_0^1u'u\,dx\ge-\dfrac12\|u'\|_2^2-\dfrac12\|u\|_2^2$.

**Passo 3 — sostituiamo nella stima di $B(u,u)$.** $B(u,u)\ge\|u'\|_2^2-\dfrac12\|u'\|_2^2-\dfrac12\|u\|_2^2+\|u\|_2^2=\dfrac12\|u'\|_2^2+\dfrac12\|u\|_2^2$.

**Passo 4 — usiamo Poincaré per riscrivere in termini della sola norma $H^1_0$.** Su $H^1_0(0,1)$, la norma naturale è $\|u\|_{H^1_0}^2=\|u'\|_2^2$ (essendo $u=0$ al bordo, come nel notebook 15.7). Dal passo 3: $B(u,u)\ge\dfrac12\|u'\|_2^2=\dfrac12\|u\|_{H^1_0}^2$ (scartando il termine aggiuntivo $\frac12\|u\|_2^2\ge0$, che può solo aiutare la disuguaglianza).

**Conclusione:** $B(u,u)\ge\dfrac12\|u\|_{H^1_0}^2$ per ogni $u\in H^1_0(0,1)$: $B$ è coerciva con costante $\alpha=1/2$. **Verifica della continuità (necessaria per Lax-Milgram, verifica rapida):** $|B(u,v)|\le\|u'\|_2\|v'\|_2+\|u'\|_2\|v\|_2+\|u\|_2\|v\|_2\le C\|u\|_{H^1_0}\|v\|_{H^1_0}$ (usando Poincaré per controllare $\|u\|_2,\|v\|_2$ con $\|u'\|_2,\|v'\|_2$). Essendo $B$ sia continua sia coerciva, per Lax-Milgram (teoria) il problema associato $-u''+u'+u=f$, $u(0)=u(1)=0$ ha un'unica soluzione debole per ogni $f\in L^2(0,1)$ — **nonostante** $B$ non sia simmetrica (il termine $u'v$ non è simmetrico in $u,v$), Riesz da solo non basterebbe, ma Lax-Milgram sì.

In [ ]:
import numpy as np
from scipy import integrate

def verifica_coercivita(u, u_prime, campione=1000):
    x = np.linspace(0, 1, campione)
    B_uu = integrate.trapezoid(u_prime(x)**2 + u_prime(x)*u(x) + u(x)**2, x)
    norma_H1_0_quadrato = integrate.trapezoid(u_prime(x)**2, x)
    return B_uu, norma_H1_0_quadrato

def u(x):
    return np.sin(np.pi*x)
def u_prime(x):
    return np.pi*np.cos(np.pi*x)

B_uu, norma_quadrato = verifica_coercivita(u, u_prime)
print(f"B(u,u) = {B_uu:.5f}")
print(f"||u||_H1_0^2 = {norma_quadrato:.5f}")
print(f"Rapporto B(u,u)/||u||^2 = {B_uu/norma_quadrato:.5f} (deve essere >= alpha = 0.5)")

## Esercizi da risolvere

1. Verifica che la forma $B(u,v)=\int_\Omega\nabla u\cdot\nabla v\,dx$ (caso del notebook 15.7) è **simmetrica**: $B(u,v)=B(v,u)$.
2. Dimostra che se $B$ è simmetrica e coerciva, allora $\sqrt{B(u,u)}$ definisce una norma equivalente a $\|u\|_{H^1_0}$ (suggerimento: usa direttamente la coercività per una direzione, e la continuità di $B$ per l'altra).
3. Spiega perché la condizione $c(x)\ge0$ nel corollario della teoria è necessaria per la coercività, trovando un controesempio con $c$ fortemente negativo dove $B(u,u)$ può diventare negativo (suggerimento: pensa a $u$ con $\|\nabla u\|_2$ piccolo ma $\|u\|_2$ grande — possibile? Rifletti su cosa dice esattamente Poincaré in questo caso).
4. *(più impegnativo)* Usa Lax-Milgram per dimostrare esistenza e unicità della soluzione debole di $-u''+\lambda u=f$ su $(0,1)$ con $u(0)=u(1)=0$, per ogni $\lambda>-\pi^2$ (suggerimento: usa la disuguaglianza di Poincaré ottimale su $(0,1)$, che si dimostra essere $\|u\|_2^2\le\dfrac1{\pi^2}\|u'\|_2^2$ — collegata al primo autovalore del laplaciano con condizioni di Dirichlet, un'anticipazione dei metodi spettrali per PDE — per garantire che $B(u,u)=\|u'\|_2^2+\lambda\|u\|_2^2$ resti coerciva anche per $\lambda$ moderatamente negativo).

<details>
<summary><strong>👉 Soluzioni (clicca per visualizzare)</strong></summary>

**1.** $B(v,u)=\int_\Omega\nabla v\cdot\nabla u\,dx=\int_\Omega\nabla u\cdot\nabla v\,dx=B(u,v)$ (il prodotto scalare $\nabla u\cdot\nabla v$ tra vettori è sempre simmetrico, essendo il prodotto scalare euclideo commutativo, notebook 6.20).

**2.** *Positività*: dalla coercività, $B(u,u)\ge\alpha\|u\|_{H^1_0}^2\ge0$, con uguaglianza solo se $u=0$ (essendo $\alpha>0$). *Omogeneità e disuguaglianza triangolare*: seguono dal fatto che $\sqrt{B(u,u)}$ è la norma indotta dal prodotto scalare $B$ stesso (essendo $B$ simmetrica e bilineare, positiva, quindi un prodotto scalare a tutti gli effetti, notebook 14.3). *Equivalenza*: dalla coercività, $\|u\|_{H^1_0}\le\dfrac1{\sqrt\alpha}\sqrt{B(u,u)}$; dalla continuità di $B$ ($B(u,u)\le C\|u\|_{H^1_0}^2$), $\sqrt{B(u,u)}\le\sqrt C\|u\|_{H^1_0}$. Le due stime, insieme, danno l'equivalenza delle norme (notebook 14.1).

**3.** Se $c(x)=-M$ (costante fortemente negativa) e prendiamo $u$ con $\|\nabla u\|_2$ piccolo, per Poincaré (notebook 15.7) $\|u\|_2\le C\|\nabla u\|_2$ è comunque **limitato dall'alto** da $\|\nabla u\|_2$ — sembrerebbe impossibile avere $\|u\|_2$ grande con $\|\nabla u\|_2$ piccolo. Tuttavia $B(u,u)=\|\nabla u\|_2^2-M\|u\|_2^2$ diventa negativo se $M$ è sufficientemente grande **rispetto alla costante ottimale di Poincaré**: prendendo $u$ vicino alla prima autofunzione del laplaciano (che rende $\|u\|_2^2/\|\nabla u\|_2^2$ il più grande possibile compatibilmente con Poincaré, un rapporto legato al primo autovalore, notebook 6.17, anticipato nell'esercizio 4), se $M$ supera questo rapporto critico, $B(u,u)<0$: la coercività fallisce per $c$ sufficientemente negativo, confermando che $c\ge0$ è una condizione sufficiente semplice, anche se non sempre stretta, necessaria in generale solo entro una soglia legata a Poincaré.

**4.** $B(u,u)=\|u'\|_2^2+\lambda\|u\|_2^2$. Usando $\|u\|_2^2\le\dfrac1{\pi^2}\|u'\|_2^2$ (Poincaré ottimale su $(0,1)$, dato): se $\lambda\ge0$, $B(u,u)\ge\|u'\|_2^2\ge0$ banalmente coerciva (anzi già con $\alpha=1$). Se $-\pi^2<\lambda<0$: $B(u,u)=\|u'\|_2^2+\lambda\|u\|_2^2\ge\|u'\|_2^2+\lambda\cdot\dfrac1{\pi^2}\|u'\|_2^2$ (usando $\lambda<0$, quindi moltiplicare per $\lambda$ **inverte** la disuguaglianza di Poincaré: $\lambda\|u\|_2^2\ge\lambda\cdot\frac1{\pi^2}\|u'\|_2^2$, essendo $\|u\|_2^2\le\frac1{\pi^2}\|u'\|_2^2$ e $\lambda$ negativo) $=\Big(1+\dfrac\lambda{\pi^2}\Big)\|u'\|_2^2$. Essendo $\lambda>-\pi^2$, si ha $1+\lambda/\pi^2>0$: dunque $B(u,u)\ge\alpha\|u'\|_2^2=\alpha\|u\|_{H^1_0}^2$ con $\alpha=1+\lambda/\pi^2>0$: $B$ è coerciva. Essendo anche continua (verifica diretta, simile all'esercizio svolto), Lax-Milgram garantisce esistenza e unicità della soluzione debole per ogni $\lambda>-\pi^2$.

</details>

## Riepilogo finale

- Un operatore $L$ è **ellittico** se la sua matrice dei coefficienti principali è uniformemente positiva definita — la condizione che generalizza $-\Delta$ (dove $A=I$) a coefficienti variabili, anche non simmetrici.
- Il **teorema di Lax-Milgram** generalizza Riesz a forme bilineari continue e **coercive**, non necessariamente simmetriche: riducendo il problema a un operatore $T$ (rappresentante Riesz di $B$) e dimostrando che $T$ è invertibile grazie alla coercività.
- Quando $B$ è simmetrica, Lax-Milgram si riduce esattamente a Riesz applicato al prodotto scalare indotto da $B$ — coerentemente con il caso particolare del notebook 15.7.
- La coercività, spesso dimostrata combinando la disuguaglianza di **Young** (per stimare termini misti) con quella di **Poincaré** (per controllare la norma $L^2$ con il gradiente), è la condizione strutturale che garantisce buona posizione per un'ampia classe di PDE ellittiche.

## Collegamenti

Il prossimo notebook è **15.9 — PDE paraboliche**: generalizziamo l'equazione del calore (notebook 15.4) a operatori ellittici generali dipendenti dal tempo, usando la teoria degli spazi di Hilbert dipendenti da un parametro (spazi di Bochner) per formulare rigorosamente il problema di evoluzione.